# 08 — End-to-End Gating Verification

**Purpose:** Verify that `src/gating.py` correctly selects sessions and produces results
consistent with the retrospective simulation in notebook 07.

**Approach:**
1. Load existing HDFS explain-all results (margins + signatures already available)
2. Build mock `ScreenerOutput` objects from the JSONL data
3. Call `gate()` directly — verify K, ordering, and signature coverage
4. Cross-check against notebook 07 simulation results (72 evaluation points)

**Key insight:** `gate()` only uses `margin` from `ScreenerOutput`. Since we already
have margins in the normalized JSONL, no re-run of screener, BM25, or LLM is needed.

## 1. Imports & Configuration

In [ ]:
import sys
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# Project root
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Verify imports
from src.gating import GatingMode, GatingConfig, gate
from src.screener import ScreenerOutput
from src.data_loader import Session

print(f"Project root: {PROJECT_ROOT}")
print(f"[OK] All imports successful")
print(f"[OK] GatingMode values: {[m.value for m in GatingMode]}")

In [ ]:
# === Load existing HDFS results and build mock objects ===
HDFS_JSONL = PROJECT_ROOT / "results_HDFS" / "explanations_HDFS_20260215_220648.normalized.jsonl"

records = []
sessions = []
screener_outputs = []

with open(HDFS_JSONL, 'r') as f:
    for line in f:
        d = json.loads(line)
        if d.get('explanation') is None:
            continue

        sid = d['session_id']
        margin = d['screener']['margin']
        prob = d['screener']['prob']

        records.append({
            'session_id': sid,
            'margin': margin,
            'uncertainty': 1.0 - margin,
            'normalized_signature': d.get('normalized_signature', 'UNKNOWN'),
            'total_tokens': d['metrics']['total_tokens'],
        })

        # Mock Session (gate() only needs session_id)
        s = Session(session_id=sid, lines=[], label=d['label'])
        sessions.append(s)

        # Real ScreenerOutput with actual margin
        so = ScreenerOutput(
            session_id=sid,
            pred=1,  # all are predicted anomalies
            logits=[0.0, 0.0],
            prob=prob,
            margin=margin,
        )
        screener_outputs.append(so)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} anomaly sessions from HDFS")
print(f"  Unique signatures: {df['normalized_signature'].nunique()}")
print(f"  Margin range: [{df['margin'].min():.6f}, {df['margin'].max():.6f}]")

## 2. Test Mode a — Explain-All (Pass-Through)

Verify `gate()` with `GatingMode.EXPLAIN_ALL` returns all anomaly sessions unchanged.

In [ ]:
# === Mode a: Explain-All — should return everything ===
config_a = GatingConfig(mode=GatingMode.EXPLAIN_ALL, budget=1.0)
result_a = gate(sessions, screener_outputs, config_a)

n_a = len(result_a)
sigs_a = set(r[1].session_id for r in result_a)  # session_ids
assert n_a == len(sessions), f"Mode a should return all {len(sessions)}, got {n_a}"

# Recover signatures via session_id lookup
sid_to_sig = dict(zip(df['session_id'], df['normalized_signature']))
sigs_mode_a = set(sid_to_sig[s.session_id] for s, _ in result_a)

print(f"[OK] Mode a returned all {n_a} sessions (pass-through)")
print(f"  Unique signatures: {len(sigs_mode_a)}")

## 3. Test Mode b — Top-K by Uncertainty (B=0.20)

Verify `gate()` with `GatingMode.TOP_K` and `budget=0.20` selects exactly K = floor(0.20 * N) sessions, and that they are the ones with the lowest margins.

In [ ]:
# === Mode b: Top-K at multiple budget levels ===
BUDGET_LEVELS = [0.10, 0.20, 0.30, 0.50, 0.75, 1.00]
N = len(sessions)

print(f"Total anomaly sessions: {N}")
print(f"{'Budget':>8} {'K':>6} {'Returned':>10} {'Sigs':>6} {'Coverage':>10} {'Max Margin':>12}")
print("-" * 60)

gate_results = {}
for B in BUDGET_LEVELS:
    config = GatingConfig(mode=GatingMode.TOP_K, budget=B)
    result = gate(sessions, screener_outputs, config)
    K_expected = max(1, math.floor(B * N))

    # Get signatures
    selected_sigs = set(sid_to_sig[s.session_id] for s, _ in result)
    coverage = len(selected_sigs) / len(sigs_mode_a)

    # Get max margin in selected set (should be lowest margins = most uncertain)
    max_margin = max(so.margin for _, so in result)

    gate_results[B] = {
        'K': K_expected,
        'returned': len(result),
        'sigs': len(selected_sigs),
        'coverage': coverage,
        'max_margin': max_margin,
        'selected_sigs': selected_sigs,
    }

    ok = len(result) == K_expected
    print(f"{B:>8.0%} {K_expected:>6} {len(result):>10} {len(selected_sigs):>6} "
          f"{coverage:>9.1%} {max_margin:>12.6f}  {'[OK]' if ok else '[FAIL]'}")

## 4. Ordering Verification

Verify that `gate()` selects sessions sorted by ascending margin (= descending uncertainty), i.e., the most uncertain sessions are selected first.

In [ ]:
# === Verify ordering: gate() should return ascending margin ===
config_b20 = GatingConfig(mode=GatingMode.TOP_K, budget=0.20)
result_b20 = gate(sessions, screener_outputs, config_b20)

margins_selected = [so.margin for _, so in result_b20]
is_sorted = all(margins_selected[i] <= margins_selected[i+1] for i in range(len(margins_selected)-1))
print(f"[{'OK' if is_sorted else 'FAIL'}] Selected sessions are sorted by ascending margin")

# Verify these are truly the K smallest margins
all_margins = sorted([so.margin for so in screener_outputs])
K = len(result_b20)
threshold_margin = all_margins[K - 1]
max_selected_margin = max(margins_selected)

print(f"  K = {K}")
print(f"  K-th smallest margin (threshold): {threshold_margin:.6f}")
print(f"  Max margin in selected set:       {max_selected_margin:.6f}")

assert max_selected_margin <= threshold_margin + 1e-10, "Gate selected a session above the threshold!"
print(f"[OK] All selected sessions have margin <= threshold")

# Show a few examples
print(f"\n  Sample selected (most uncertain):")
for s, so in result_b20[:5]:
    sig = sid_to_sig[s.session_id]
    print(f"    {s.session_id[:20]:>20}  margin={so.margin:.6f}  u={1-so.margin:.6f}  sig={sig}")

## 5. Cross-Check with Notebook 07 Simulation

Compare `gate()` results with the retrospective simulation from `gating_simulation_results.json`.
Both should produce identical signature coverage at each budget level.

In [ ]:
# === Cross-check with notebook 07 retrospective simulation ===
sim_path = PROJECT_ROOT / "results" / "gating_simulation_results.json"
sim_results = pd.read_json(sim_path)

# Filter HDFS Uncertainty strategy
sim_hdfs_unc = sim_results[
    (sim_results['dataset'] == 'HDFS') &
    (sim_results['strategy'] == 'Uncertainty')
].set_index('budget')

print("=" * 70)
print("Cross-Check: gate() vs Notebook 07 Retrospective Simulation")
print("=" * 70)
print(f"{'Budget':>8} {'gate() Cov':>12} {'Sim Cov':>12} {'gate() Sigs':>12} {'Sim Sigs':>10} {'Match':>7}")
print("-" * 70)

all_match = True
for B in BUDGET_LEVELS:
    gate_cov = gate_results[B]['coverage']
    gate_sigs = gate_results[B]['sigs']

    if B in sim_hdfs_unc.index:
        sim_cov = sim_hdfs_unc.loc[B, 'coverage']
        sim_sigs = int(sim_hdfs_unc.loc[B, 'unique_sigs'])
        match = abs(gate_cov - sim_cov) < 1e-6
        if not match:
            all_match = False
        print(f"{B:>8.0%} {gate_cov:>11.1%} {sim_cov:>11.1%} "
              f"{gate_sigs:>12} {sim_sigs:>10} {'[OK]' if match else '[DIFF]':>7}")
    else:
        print(f"{B:>8.0%} {gate_cov:>11.1%} {'N/A':>12} {gate_sigs:>12} {'N/A':>10}")

print()
if all_match:
    print("[OK] gate() produces IDENTICAL coverage to notebook 07 simulation")
    print("     src/gating.py is verified consistent with retrospective analysis")
else:
    print("[WARN] Some coverage values differ -- investigate")

# Highlight B=0.20 operating point
print(f"\n--- Operating Point: B=0.20 ---")
g = gate_results[0.20]
print(f"  Sessions selected: {g['returned']} / {N}")
print(f"  Signature coverage: {g['coverage']:.1%} ({g['sigs']}/{len(sigs_mode_a)})")
missed = sigs_mode_a - g['selected_sigs']
if missed:
    print(f"  Missing signatures: {sorted(missed)}")
else:
    print(f"  [OK] All signatures covered")